# Road Damage Object Detection — YOLO11s

Classes: **Crack**, **Manhole**, **Pothole** · Dataset: Roboflow `project-dvd12/road-damage-eul7g` **v3** (fixed 70/20/10 split, Fit 640×640, 802 unlabeled null images excluded, no offline augmentation).

Same dataset, split, image size, epochs and hyperparameters as the other two notebooks, so the results are directly comparable.

Runs on **Kaggle** or **Google Colab** (detected automatically). Use the same platform and GPU type for all three models, otherwise speed/FPS numbers are not comparable.

**Kaggle:**
1. Settings → Accelerator → **GPU T4 x2** (one GPU is used), and Settings → **Internet on**.
2. Add-ons → **Secrets** → add `ROBOFLOW_API_KEY` (your Roboflow key) and attach it to the notebook.
3. Save Version → **Save & Run All (Commit)**. It runs in the background (browser can be closed); results land in the notebook **Output** (`/kaggle/working/road_damage/`).

**Colab:**
1. Runtime → Change runtime type → **T4 GPU**.
2. Add `ROBOFLOW_API_KEY` in Colab **Secrets** (🔑) and enable notebook access.
3. Run all. Everything is saved to Google Drive, so a disconnect does not lose the run (see the `RESUME` flag).

## 1. Environment

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q -U ultralytics roboflow

import torch, ultralytics
ultralytics.checks()
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - change runtime to T4!'}")

## 2. Config (Kaggle output folder or Google Drive)

In [ ]:
import os

ON_KAGGLE = os.path.exists("/kaggle/working")
if ON_KAGGLE:
    BASE_DIR = "/kaggle/working/road_damage"
else:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/road_damage"

MODEL_WEIGHTS = "yolo11s.pt"
RUN_NAME = "yolo11s"
PROJECT_DIR = f"{BASE_DIR}/runs_v3"
RUN_DIR = f"{PROJECT_DIR}/{RUN_NAME}"
BEST_PT = f"{RUN_DIR}/weights/best.pt"
IMGSZ = 640
SEED = 42

os.makedirs(PROJECT_DIR, exist_ok=True)
print("Run dir:", RUN_DIR)

## 3. Download dataset from Roboflow (v3, fixed 70/20/10 split)

The split is stored inside Roboflow version 3, so every notebook gets exactly the same train/valid/test images. Needs the `ROBOFLOW_API_KEY` secret.

In [ ]:
import glob, shutil, yaml
from roboflow import Roboflow

ROBOFLOW_VERSION = 3  # 70/20/10 split, Fit 640x640, 'unlabeled' null images excluded, no offline augmentation

# Roboflow API key comes from a secret named ROBOFLOW_API_KEY (never hard-code it)
if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    api_key = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
else:
    from google.colab import userdata
    api_key = userdata.get("ROBOFLOW_API_KEY")

DATA_ROOT = "/tmp/road_damage"
shutil.rmtree(DATA_ROOT, ignore_errors=True)
rf = Roboflow(api_key=api_key)
dataset = rf.workspace("project-dvd12").project("road-damage-eul7g") \
            .version(ROBOFLOW_VERSION).download("yolov11", location=DATA_ROOT)

# Rewrite data.yaml with absolute paths so train/valid/test always resolve
DATA = f"{DATA_ROOT}/data.yaml"
with open(DATA) as f:
    data_cfg = yaml.safe_load(f)
data_cfg.update(path=DATA_ROOT, train="train/images", val="valid/images", test="test/images")
with open(DATA, "w") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)
NAMES = data_cfg["names"]

for split in ["train", "valid", "test"]:
    counts = {n: 0 for n in NAMES}
    for lbl in glob.glob(f"{DATA_ROOT}/{split}/labels/*.txt"):
        for line in open(lbl).read().split("\n"):
            if line.strip():
                counts[NAMES[int(float(line.split()[0]))]] += 1
    n_img = len(os.listdir(f"{DATA_ROOT}/{split}/images"))
    print(f"{split:<6} {n_img:>5} images | " + " | ".join(f"{k}: {v}" for k, v in counts.items()))
print(open(DATA).read())

## 4. Training config

Identical in all three notebooks. `USE_ULTRALYTICS_AUG = True` keeps Ultralytics online augmentation (mosaic, flip, HSV, scale, translate).
Roboflow v3 has no offline augmentation, so keep it `True`; with `False` the model trains on raw images only.

In [ ]:
from ultralytics import YOLO

USE_ULTRALYTICS_AUG = True

AUG = dict(hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, degrees=0.0, translate=0.1,
           scale=0.5, fliplr=0.5, mosaic=1.0, mixup=0.0, close_mosaic=10)
if not USE_ULTRALYTICS_AUG:
    AUG = {k: 0.0 for k in AUG} | {"close_mosaic": 0}

TRAIN_ARGS = dict(
    data=DATA,
    epochs=100,
    imgsz=IMGSZ,
    batch=16,
    device=0,
    workers=2,
    patience=20,
    seed=SEED,
    deterministic=True,
    cache=False,  # cache="ram" can run Colab out of RAM and crash the kernel
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,
    project=PROJECT_DIR,
    name=RUN_NAME,
    exist_ok=True,
    save_period=10,
    plots=True,
    **AUG,
)
TRAIN_ARGS

## 5. Train

In [ ]:
RESUME = False  # set True after a Colab disconnect to continue from last.pt
last_pt = f"{RUN_DIR}/weights/last.pt"

if RESUME and os.path.exists(last_pt):
    model = YOLO(last_pt)
    model.train(resume=True)
else:
    model = YOLO(MODEL_WEIGHTS)
    model.train(**TRAIN_ARGS)

print("best.pt exists:", os.path.exists(BEST_PT))

# Free the GPU memory held by the training run, otherwise evaluation below can hit CUDA OOM
import gc
del model
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory in use after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
from IPython.display import Image, display

display(Image(f"{RUN_DIR}/results.png", width=1000))

## 6. Test-set metrics: mAP, Precision, Recall (overall + per class)

In [ ]:
best = YOLO(BEST_PT)
metrics = best.val(data=DATA, split="test", imgsz=IMGSZ, batch=16, plots=True,
                   project=PROJECT_DIR, name=f"{RUN_NAME}_test", exist_ok=True)

names = metrics.names
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

per_class = {}
print(f"\n{'class':<10}{'P':>8}{'R':>8}{'AP50':>8}{'AP50-95':>9}")
for i, c in enumerate(metrics.box.ap_class_index):
    p, r, ap50, ap = metrics.box.class_result(i)
    per_class[names[c]] = dict(precision=float(p), recall=float(r), ap50=float(ap50), ap50_95=float(ap))
    print(f"{names[c]:<10}{p:>8.3f}{r:>8.3f}{ap50:>8.3f}{ap:>9.3f}")

In [ ]:
test_dir = f"{PROJECT_DIR}/{RUN_NAME}_test"
display(Image(f"{test_dir}/confusion_matrix_normalized.png", width=600))

## 7. Mean IoU

Ultralytics does not report IoU directly. Each prediction (conf ≥ 0.25) is matched greedily to the unmatched ground-truth box of the same class with the highest IoU (≥ 0.5).
**Mean IoU** = average IoU over these true-positive matches (localization quality).

In [ ]:
import glob
import numpy as np

TEST_IMAGES = sorted(glob.glob(f"{DATA_ROOT}/test/images/*"))

def load_gt(img_path, w, h):
    label_path = os.path.splitext(img_path.replace("/images/", "/labels/"))[0] + ".txt"
    cls, boxes = [], []
    if os.path.exists(label_path):
        for line in open(label_path).read().strip().splitlines():
            v = list(map(float, line.split()))
            if len(v) == 5:
                _, xc, yc, bw, bh = v
                x1, y1, x2, y2 = xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2
            else:  # polygon label -> bounding box
                xs, ys = v[1::2], v[2::2]
                x1, y1, x2, y2 = min(xs), min(ys), max(xs), max(ys)
            cls.append(int(v[0]))
            boxes.append([x1 * w, y1 * h, x2 * w, y2 * h])
    return np.array(cls, dtype=int), np.array(boxes, dtype=float).reshape(-1, 4)

def iou_matrix(a, b):
    tl = np.maximum(a[:, None, :2], b[None, :, :2])
    br = np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.prod(np.clip(br - tl, 0, None), axis=2)
    area_a = np.prod(a[:, 2:] - a[:, :2], axis=1)
    area_b = np.prod(b[:, 2:] - b[:, :2], axis=1)
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)

tp_ious = {n: [] for n in names.values()}
n_gt = 0
# Folder source + batch=1: a Python list is loaded as ONE batch (all test images at once -> CUDA OOM)
for r in best.predict(source=f"{DATA_ROOT}/test/images", batch=1, imgsz=IMGSZ, conf=0.25, stream=True, verbose=False):
    h, w = r.orig_shape
    gt_cls, gt_box = load_gt(r.path, w, h)
    n_gt += len(gt_cls)
    if len(r.boxes) == 0 or len(gt_cls) == 0:
        continue
    order = r.boxes.conf.cpu().numpy().argsort()[::-1]
    pr_box = r.boxes.xyxy.cpu().numpy()[order]
    pr_cls = r.boxes.cls.cpu().numpy().astype(int)[order]
    ious = iou_matrix(pr_box, gt_box)
    matched = np.zeros(len(gt_cls), dtype=bool)
    for i in range(len(pr_box)):
        cand = np.where((gt_cls == pr_cls[i]) & ~matched)[0]
        if len(cand) == 0:
            continue
        j = cand[ious[i, cand].argmax()]
        if ious[i, j] >= 0.5:
            matched[j] = True
            tp_ious[names[pr_cls[i]]].append(ious[i, j])

all_ious = [v for vals in tp_ious.values() for v in vals]
mean_iou = float(np.mean(all_ious)) if all_ious else 0.0
mean_iou_per_class = {k: float(np.mean(v)) if v else 0.0 for k, v in tp_ious.items()}
print(f"Mean IoU (TP matches): {mean_iou:.4f}   [{len(all_ious)} TPs / {n_gt} GT boxes]")
for k, v in mean_iou_per_class.items():
    print(f"  {k:<10} {v:.4f}")

## 8. Inference speed / FPS (batch = 1) + model size

In [ ]:
from ultralytics.utils.torch_utils import get_flops

def measure_speed(weights, images, device, n=100, warmup=10):
    m = YOLO(weights, task="detect")  # fresh model per device
    images = images[:n]
    for img in images[:warmup]:
        m.predict(img, imgsz=IMGSZ, device=device, verbose=False)
    pre = inf = post = 0.0
    for img in images:
        s = m.predict(img, imgsz=IMGSZ, device=device, verbose=False)[0].speed
        pre, inf, post = pre + s["preprocess"], inf + s["inference"], post + s["postprocess"]
    k = len(images)
    total = (pre + inf + post) / k
    return dict(preprocess_ms=pre / k, inference_ms=inf / k, postprocess_ms=post / k,
                total_ms=total, fps=1000 / total)

gpu_speed = measure_speed(BEST_PT, TEST_IMAGES, device=0, n=len(TEST_IMAGES))
cpu_speed = measure_speed(BEST_PT, TEST_IMAGES, device="cpu", n=50)

size_mb = os.path.getsize(BEST_PT) / 1e6
params_m = sum(p.numel() for p in best.model.parameters()) / 1e6
gflops = get_flops(best.model, IMGSZ)

print(f"GPU (T4): {gpu_speed['total_ms']:.1f} ms/img -> {gpu_speed['fps']:.1f} FPS  (inference only {gpu_speed['inference_ms']:.1f} ms)")
print(f"CPU     : {cpu_speed['total_ms']:.1f} ms/img -> {cpu_speed['fps']:.1f} FPS")
print(f"best.pt : {size_mb:.1f} MB | params: {params_m:.2f} M | GFLOPs: {gflops:.1f}")

## 9. Deployment efficiency: ONNX export → size, CPU speed, accuracy check

In [ ]:
onnx_path = YOLO(BEST_PT).export(format="onnx", imgsz=IMGSZ, simplify=True)
onnx_size_mb = os.path.getsize(onnx_path) / 1e6
onnx_cpu_speed = measure_speed(onnx_path, TEST_IMAGES, device="cpu", n=50)
onnx_metrics = YOLO(onnx_path, task="detect").val(data=DATA, split="test", imgsz=IMGSZ, batch=1,
                                                 device="cpu", plots=False, verbose=False,
                                                 project=PROJECT_DIR, name=f"{RUN_NAME}_onnx_test", exist_ok=True)

print(f"ONNX: {onnx_path}")
print(f"ONNX size: {onnx_size_mb:.1f} MB")
print(f"ONNX CPU : {onnx_cpu_speed['total_ms']:.1f} ms/img -> {onnx_cpu_speed['fps']:.1f} FPS")
print(f"ONNX mAP50-95 (test): {onnx_metrics.box.map:.4f}  (PyTorch: {metrics.box.map:.4f})")

## 10. Save summary (used later by the comparison notebook)

In [ ]:
import json
import pandas as pd

summary = dict(
    model=RUN_NAME,
    map50=float(metrics.box.map50),
    map50_95=float(metrics.box.map),
    precision=float(metrics.box.mp),
    recall=float(metrics.box.mr),
    mean_iou=mean_iou,
    gpu_ms=gpu_speed["total_ms"],
    gpu_fps=gpu_speed["fps"],
    cpu_ms=cpu_speed["total_ms"],
    cpu_fps=cpu_speed["fps"],
    size_mb=size_mb,
    params_m=params_m,
    gflops=gflops,
    onnx_size_mb=onnx_size_mb,
    onnx_cpu_fps=onnx_cpu_speed["fps"],
    onnx_map50_95=float(onnx_metrics.box.map),
    per_class=per_class,
    mean_iou_per_class=mean_iou_per_class,
    train_args={k: v for k, v in TRAIN_ARGS.items() if k != "data"},
)

with open(f"{PROJECT_DIR}/{RUN_NAME}_results.json", "w") as f:
    json.dump(summary, f, indent=2)

pd.Series({k: v for k, v in summary.items() if not isinstance(v, dict)}).to_frame(RUN_NAME)

## 11. Sample predictions on test images

In [ ]:
import matplotlib.pyplot as plt

sample = TEST_IMAGES[:8]
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, img_path in zip(axes.flat, sample):
    pred = best.predict(source=img_path, imgsz=IMGSZ, conf=0.35, verbose=False)[0]
    ax.imshow(pred.plot()[:, :, ::-1])
    ax.axis("off")
plt.suptitle(RUN_NAME, fontsize=16)
plt.tight_layout()
plt.show()

## 12. (Optional) Video inference

In [ ]:
VIDEO_URL = ""  # e.g. a YouTube dashcam video of a road

if VIDEO_URL:
    !pip install -q yt-dlp
    !yt-dlp -f mp4 -o video.mp4 "{VIDEO_URL}"
    best.predict(source="video.mp4", imgsz=IMGSZ, conf=0.4, save=True, vid_stride=2,
                 project=PROJECT_DIR, name=f"{RUN_NAME}_video", exist_ok=True)